# LeetCode 3579 — Minimum Steps to Convert String with Operations bằng Python

- Link: https://leetcode.com/problems/minimum-steps-to-convert-string-with-operations/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động chia đoạn** (partition DP), **đếm cặp ký tự lệch**, **ghép cặp hoán đổi**, kỹ thuật **mở rộng từ tâm** cho đoạn bị đảo ngược

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minOperations(self, word1: str, word2: str) -> int:
```

## 1. Mục tiêu học

1. Hiểu ràng buộc "mỗi vị trí tham gia **tối đa một** lần thay, một lần hoán đổi, một lần đảo" và suy ra **chi phí của một đoạn** có công thức đóng.
2. Tính chi phí đoạn không đảo: `số vị trí lệch − số cặp hoán đổi được`, trong đó cặp hoán đổi là hai vị trí lệch "ngược nhau" `(x→y)` và `(y→x)`.
3. Với đoạn được đảo: cộng thêm 1 thao tác và so `reverse(a)` với `b`.
4. Viết **DP chia đoạn** `dp[j] = min(dp[i] + cost(i, j))` và tối ưu việc tính `cost` từ `O(n^3)` xuống `O(n^2)` (mở rộng sang phải / mở rộng từ tâm).
5. Kiểm chứng công thức bằng **BFS vét cạn mọi chuỗi thao tác** trên chuỗi ngắn.

## 2. Đề bài & ví dụ

Cho hai chuỗi `word1`, `word2` cùng độ dài. Cần biến `word1` thành `word2`.

Chia `word1` thành một hay nhiều **đoạn liên tiếp**. Trên mỗi đoạn được dùng ba loại thao tác (mỗi thao tác tính 1):

1. **Thay**: đổi ký tự ở một vị trí thành chữ cái khác.
2. **Hoán đổi**: đổi chỗ hai ký tự trong đoạn.
3. **Đảo**: đảo ngược cả đoạn.

Mỗi vị trí của đoạn chỉ được tham gia **tối đa một** lần thay, **tối đa một** lần hoán đổi và **tối đa một** lần đảo. Trả về tổng số thao tác ít nhất.

**Ví dụ 1:** `"abcdf" → "dacbe"`: `4` (chia `"ab" | "c" | "df"`: đảo + thay; không làm gì; thay + thay).

**Ví dụ 2:** `"abceded" → "baecfef"`: `4`.

**Ví dụ 3:** `"abcdef" → "fedabc"`: `2` (đảo cả chuỗi thành `"fedcba"`, rồi hoán đổi để được `"fedabc"`... cụ thể hoán đổi cụm cuối).

**Ràng buộc:** `1 ≤ n ≤ 100`, chỉ chữ thường.

In [1]:
examples = [
    ("abcdf", "dacbe", 4),
    ("abceded", "baecfef", 4),
    ("abcdef", "fedabc", 2),
]
for a, b, exp in examples:
    print(f"{a!r} -> {b!r}: kỳ vọng {exp}")

'abcdf' -> 'dacbe': kỳ vọng 4
'abceded' -> 'baecfef': kỳ vọng 4
'abcdef' -> 'fedabc': kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Cắt chuỗi và đảo chuỗi

`s[i:j]` là đoạn từ `i` đến `j − 1`; `s[::-1]` là chuỗi đảo ngược. `zip(a, b)` ghép từng cặp ký tự cùng vị trí.

In [2]:
s = "abcdef"
print(s[1:4], s[::-1], s[1:4][::-1])
print(list(zip("abc", "bca")))

bcd fedcba dcb
[('a', 'b'), ('b', 'c'), ('c', 'a')]


### 3.2. `Counter` đếm cặp

`collections.Counter` đếm số lần xuất hiện. Ta sẽ đếm cặp lệch `(x, y)` nghĩa là "ở vị trí này `a` có `x` nhưng cần `y`".

In [3]:
from collections import Counter
a, b = "abab", "baba"
pairs = Counter((x, y) for x, y in zip(a, b) if x != y)
print(pairs)

Counter({('a', 'b'): 2, ('b', 'a'): 2})


### 3.3. Quy hoạch động chia đoạn

Mẫu chung: `dp[j]` = chi phí tốt nhất cho tiền tố độ dài `j`; đoạn cuối là `[i, j)`:

`dp[0] = 0`, `dp[j] = min over i < j của dp[i] + cost(i, j)`.

Độ phức tạp `O(n^2)` lần gọi `cost`.

In [4]:
def partition_dp(n, cost):
    dp = [0] + [float('inf')] * n
    for j in range(1, n + 1):
        for i in range(j):
            dp[j] = min(dp[j], dp[i] + cost(i, j))
    return dp[n]

# ví dụ đồ chơi: chi phí đoạn = (độ dài - 2)^2 + 1
print(partition_dp(6, lambda i, j: (j - i - 2) ** 2 + 1))

3


### 3.4. Mảng phẳng thay cho `dict` cặp chữ cái

Cặp `(x, y)` với `x, y ∈ 'a'..'z'` mã hóa thành `26·x + y` (676 ô). Truy cập mảng nhanh hơn `dict` khi phải cập nhật nhiều lần.

In [5]:
def code(x, y):
    return (ord(x) - 97) * 26 + (ord(y) - 97)
print(code('a', 'b'), code('b', 'a'), code('z', 'z'))

1 26 675


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xét **một đoạn** `a` cần biến thành `b` (không đảo). Ở mỗi vị trí lệch `a[i] ≠ b[i]`:

- Cách rẻ nhất cho từng vị trí là **thay**: 1 thao tác / vị trí.
- Nhưng nếu có hai vị trí `i, j` với `a[i] = b[j]` và `a[j] = b[i]` (tức cặp `(x→y)` và `(y→x)`), **một lần hoán đổi** sửa cả hai — tiết kiệm 1.
- Mỗi vị trí chỉ hoán đổi được một lần ⇒ các cặp hoán đổi **rời nhau**. Số cặp tối đa với loại `{x, y}` là `min(cnt[x→y], cnt[y→x])`.

⇒ `f(a, b) = (#lệch) − Σ_{x<y} min(cnt[x→y], cnt[y→x])`.

Nếu **đảo** đoạn trước: chi phí `1 + f(reverse(a), b)`.

Tính tay ví dụ 1 với cách chia `"ab" | "c" | "df"`:

| đoạn | không đảo | đảo |
|---|---|---|
| `"ab"→"da"` | lệch (a→d),(b→a): 2 − 0 = 2 | `"ba"→"da"`: 1 lệch ⇒ 1 + 1 = **2** |
| `"c"→"c"` | **0** | 1 + 0 |
| `"df"→"be"` | **2** | `"fd"→"be"`: 1 + 2 |

Tổng `2 + 0 + 2 = 4`.

In [6]:
def f_segment(a, b):
    cnt = Counter((x, y) for x, y in zip(a, b) if x != y)
    mismatches = sum(cnt.values())
    swaps = sum(min(v, cnt[(y, x)]) for (x, y), v in cnt.items() if x < y)
    return mismatches - swaps

def cost_segment(a, b):
    return min(f_segment(a, b), 1 + f_segment(a[::-1], b))

for a, b in [("ab", "da"), ("c", "c"), ("df", "be"), ("abcdef", "fedabc")]:
    print(a, b, '| không đảo:', f_segment(a, b), '| đảo:', 1 + f_segment(a[::-1], b),
          '| cost:', cost_segment(a, b))

ab da | không đảo: 2 | đảo: 2 | cost: 2
c c | không đảo: 0 | đảo: 1 | cost: 0
df be | không đảo: 2 | đảo: 3 | cost: 2
abcdef fedabc | không đảo: 5 | đảo: 2 | cost: 2


## 5. Cách 1 — Brute Force: BFS mọi chuỗi thao tác + thử mọi cách chia

Để **không phải tin** công thức ở mục 4, ta vét cạn thật sự: với một đoạn, BFS trên trạng thái `(chuỗi hiện tại, tập vị trí đã thay, tập vị trí đã hoán đổi, đã đảo chưa)`, mỗi cạnh là một thao tác hợp lệ. Số bước BFS nhỏ nhất tới `b` là chi phí thật của đoạn. Sau đó DP chia đoạn bằng chi phí này.

Chỉ chạy được với chuỗi rất ngắn và bảng chữ cái nhỏ (ký tự thay chỉ lấy trong các chữ cái xuất hiện ở `b` là đủ).

In [7]:
from collections import deque
from functools import lru_cache

@lru_cache(maxsize=None)
def seg_bfs(a, b):
    L = len(a)
    letters = sorted(set(b))
    start = (a, 0, 0, False)
    dq = deque([(start, 0)])
    seen = {start}
    while dq:
        (s, used_rep, used_swp, rev), d = dq.popleft()
        if s == b:
            return d
        nxt = []
        for i in range(L):                              # thay
            if not used_rep >> i & 1:
                for ch in letters:
                    if ch != s[i]:
                        nxt.append((s[:i] + ch + s[i + 1:], used_rep | 1 << i, used_swp, rev))
        for i in range(L):                              # hoán đổi
            for j in range(i + 1, L):
                if not used_swp >> i & 1 and not used_swp >> j & 1:
                    t = list(s)
                    t[i], t[j] = t[j], t[i]
                    nxt.append((''.join(t), used_rep, used_swp | 1 << i | 1 << j, rev))
        if not rev:                                     # đảo
            nxt.append((s[::-1], used_rep, used_swp, True))
        for st in nxt:
            if st not in seen:
                seen.add(st)
                dq.append((st, d + 1))
    return float('inf')

def min_ops_brute(w1, w2):
    n = len(w1)
    return partition_dp(n, lambda i, j: seg_bfs(w1[i:j], w2[i:j]))

print(min_ops_brute("abcdf", "dacbe"), min_ops_brute("abcab", "bacba"))

4 1


**Độ phức tạp:** số trạng thái BFS bùng nổ theo `|Σ|^L · 4^L`; chỉ dùng cho `n ≤ 6`.

**Nút thắt:** BFS trên chuỗi. Bước đột phá là nhận ra chi phí một đoạn có **công thức đóng** (mục 6), và các vị trí độc lập nhau trừ các cặp hoán đổi.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề chi phí đoạn.** Với đoạn không đảo, chi phí tối thiểu đúng bằng `f(a, b) = M − P`, trong đó `M` là số vị trí lệch và `P` là số cặp hoán đổi rời nhau tối đa (`P = Σ min(cnt[x→y], cnt[y→x])`).

- **Cận trên:** hoán đổi `P` cặp (mỗi cặp sửa 2 vị trí bằng 1 thao tác), thay `M − 2P` vị trí còn lại ⇒ `P + (M − 2P) = M − P`.
- **Cận dưới:** mỗi vị trí lệch phải được "chạm" bởi thay hoặc hoán đổi. Một thao tác thay sửa ≤ 1 vị trí. Một hoán đổi chạm 2 vị trí và sửa được **cả hai** chỉ khi chúng là cặp ngược nhau; nếu không, nó sửa tối đa 1 vị trí (và vị trí kia vẫn cần thay — mà vì mỗi vị trí chỉ thay một lần, không có lợi). Vậy với `t` thao tác "sửa đôi" (≤ P), cần ít nhất `M − 2t + t = M − t ≥ M − P` thao tác.

**Đảo.** Đảo chỉ dùng được một lần mỗi đoạn và áp lên cả đoạn. Thứ tự đảo trước hay sau không quan trọng về số lượng (đảo sau tương đương với đảo trước rồi làm các thao tác ở vị trí đối xứng), nên chi phí = `min(f(a, b), 1 + f(rev(a), b))`.

**DP chia đoạn** trên `cost(i, j)`.

Quy tắc ngắn:

- **Nếu** hai vị trí lệch là `(x→y)` và `(y→x)` **thì** ghép thành một hoán đổi.
- **Nếu** vị trí lệch không ghép được **thì** thay.
- **Nếu** đảo giúp giảm hơn 1 thao tác **thì** đảo.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP chia đoạn + tính `cost` trực tiếp bằng `Counter` (O(n^3))

Với `n ≤ 100` đây đã đủ nhanh: `~n^2/2 = 5000` đoạn, mỗi đoạn `O(n)`.

In [8]:
def min_ops_cubic(w1, w2):
    n = len(w1)
    return partition_dp(n, lambda i, j: cost_segment(w1[i:j], w2[i:j]))

for a, b, exp in examples:
    print(min_ops_cubic(a, b), '| kỳ vọng', exp)

4 | kỳ vọng 4
4 | kỳ vọng 4
2 | kỳ vọng 2


**Độ phức tạp:** Time `O(n^3)`, Space `O(n)`.

### 7.2. Cách 3 — Đệ quy có nhớ (top-down)

Cùng công thức nhưng viết dạng "đoạn đầu tiên bắt đầu ở `i`, kết thúc ở `j`": `g(i) = min_j cost(i, j) + g(j)`. Hữu ích để tư duy, kết quả giống hệt.

In [9]:
def min_ops_memo(w1, w2):
    n = len(w1)

    @lru_cache(maxsize=None)
    def g(i):
        if i == n:
            return 0
        return min(cost_segment(w1[i:j], w2[i:j]) + g(j) for j in range(i + 1, n + 1))

    return g(0)

for a, b, exp in examples:
    print(min_ops_memo(a, b), '| kỳ vọng', exp)

4 | kỳ vọng 4
4 | kỳ vọng 4
2 | kỳ vọng 2


### 7.3. Cách 4 — Bảng chi phí `O(n^2)` (lời giải chính)

**Không đảo:** cố định `i`, mở rộng `j` sang phải, mỗi lần thêm một cặp `(a[j], b[j])`. Duy trì `unmatched[x→y]` (số cặp lệch chưa được ghép) và `f`:

- nếu `x = y`: không đổi;
- nếu còn `unmatched[y→x] > 0`: ghép thành hoán đổi ⇒ `unmatched[y→x] −= 1`, `f` không đổi (thay 1 lần thay bằng 1 lần hoán đổi sửa hai vị trí);
- ngược lại `unmatched[x→y] += 1`, `f += 1`.

Ghép tham lam ngay khi gặp cho đúng `min(cnt[x→y], cnt[y→x])`.

**Có đảo:** đoạn `[i, j]` đảo thì vị trí `k` nhận ký tự `a[i + j − k]`. Cố định **tâm** `c = i + j`: khi mở rộng `i −= 1, j += 1` ta thêm hai cặp `(a[j], b[i])` và `(a[i], b[j])`. Duyệt mọi tâm (lẻ và chẵn) ⇒ `O(n^2)`.

In [10]:
def cost_tables(w1, w2):
    n = len(w1)
    A = [ord(ch) - 97 for ch in w1]
    B = [ord(ch) - 97 for ch in w2]
    fwd = [[0] * (n + 1) for _ in range(n + 1)]   # fwd[i][j]: đoạn [i, j) không đảo
    rev = [[0] * (n + 1) for _ in range(n + 1)]   # rev[i][j]: đoạn [i, j) đã đảo (chưa cộng 1)

    def add(unmatched, x, y):
        if x == y:
            return 0
        if unmatched[y * 26 + x]:
            unmatched[y * 26 + x] -= 1
            return 0
        unmatched[x * 26 + y] += 1
        return 1

    for i in range(n):
        un, f = [0] * 676, 0
        for j in range(i, n):
            f += add(un, A[j], B[j])
            fwd[i][j + 1] = f

    for c in range(2 * n - 1):                    # tâm: c = i + j (j là chỉ số cuối)
        i, j = c // 2, (c + 1) // 2
        un, f = [0] * 676, 0
        while i >= 0 and j < n:
            if i == j:
                f += add(un, A[i], B[i])
            else:
                f += add(un, A[j], B[i])
                f += add(un, A[i], B[j])
            rev[i][j + 1] = f
            i -= 1
            j += 1
    return fwd, rev

def min_ops_fast(w1, w2):
    n = len(w1)
    fwd, rev = cost_tables(w1, w2)
    dp = [0] + [10**9] * n
    for j in range(1, n + 1):
        for i in range(j):
            c = min(fwd[i][j], 1 + rev[i][j])
            if dp[i] + c < dp[j]:
                dp[j] = dp[i] + c
    return dp[n]

for a, b, exp in examples:
    print(min_ops_fast(a, b), '| kỳ vọng', exp)

4 | kỳ vọng 4
4 | kỳ vọng 4
2 | kỳ vọng 2


**Độ phức tạp:** Time `O(n^2)` (bảng) + `O(n^2)` (DP); Space `O(n^2)` cho hai bảng (có thể giảm, nhưng `n ≤ 100` thì không cần).

In [11]:
# Kiểm tra bảng O(n^2) khớp công thức trực tiếp trên mọi đoạn
import random
random.seed(1)
for _ in range(200):
    n = random.randint(1, 9)
    a = ''.join(random.choice('abc') for _ in range(n))
    b = ''.join(random.choice('abc') for _ in range(n))
    fwd, rev = cost_tables(a, b)
    for i in range(n):
        for j in range(i + 1, n + 1):
            assert fwd[i][j] == f_segment(a[i:j], b[i:j])
            assert rev[i][j] == f_segment(a[i:j][::-1], b[i:j])
print('Bảng fwd/rev khớp f_segment trên mọi đoạn.')

Bảng fwd/rev khớp f_segment trên mọi đoạn.


## 8. TRACE từng bước

Trace DP trên ví dụ 1 (`"abcdf" → "dacbe"`): in `dp[j]` và đoạn cuối tốt nhất. Sau đó trace việc mở rộng từ tâm cho một đoạn đảo.

In [12]:
def trace_dp(w1, w2):
    n = len(w1)
    fwd, rev = cost_tables(w1, w2)
    dp = [0] + [10**9] * n
    choice = [None] * (n + 1)
    for j in range(1, n + 1):
        for i in range(j):
            c_plain, c_rev = fwd[i][j], 1 + rev[i][j]
            c = min(c_plain, c_rev)
            if dp[i] + c < dp[j]:
                dp[j] = dp[i] + c
                choice[j] = (i, 'đảo' if c_rev < c_plain else 'không đảo', c)
        i, how, c = choice[j]
        print(f"dp[{j}] = {dp[j]:>2}  | đoạn cuối {w1[i:j]!r}->{w2[i:j]!r} ({how}, chi phí {c}), dp[{i}] = {dp[i]}")
    segs, j = [], n
    while j:
        i, how, c = choice[j]
        segs.append((w1[i:j], w2[i:j], how, c))
        j = i
    print('Cách chia tối ưu:', segs[::-1])
    return dp[n]

trace_dp("abcdf", "dacbe")

dp[1] =  1  | đoạn cuối 'a'->'d' (không đảo, chi phí 1), dp[0] = 0
dp[2] =  2  | đoạn cuối 'ab'->'da' (không đảo, chi phí 2), dp[0] = 0
dp[3] =  2  | đoạn cuối 'abc'->'dac' (không đảo, chi phí 2), dp[0] = 0
dp[4] =  3  | đoạn cuối 'abcd'->'dacb' (không đảo, chi phí 3), dp[0] = 0
dp[5] =  4  | đoạn cuối 'abcdf'->'dacbe' (không đảo, chi phí 4), dp[0] = 0
Cách chia tối ưu: [('abcdf', 'dacbe', 'không đảo', 4)]


4

In [13]:
def trace_center(w1, w2, c):
    n = len(w1)
    i, j = c // 2, (c + 1) // 2
    un, f = Counter(), 0
    print(f"Tâm c = {c}: bắt đầu i={i}, j={j}")
    while i >= 0 and j < n:
        added = [(w1[i], w2[i])] if i == j else [(w1[j], w2[i]), (w1[i], w2[j])]
        for x, y in added:
            if x == y:
                note = 'khớp'
            elif un[(y, x)]:
                un[(y, x)] -= 1
                note = f'ghép hoán đổi với ({y}→{x})'
            else:
                un[(x, y)] += 1
                f += 1
                note = 'chưa ghép, +1'
            print(f"   thêm cặp ({x}→{y}): {note}")
        print(f"  đoạn [{i},{j}] {w1[i:j+1]!r} đảo thành {w1[i:j+1][::-1]!r} so với {w2[i:j+1]!r}: f = {f}")
        i -= 1
        j += 1

trace_center("abcdef", "fedabc", 5)

Tâm c = 5: bắt đầu i=2, j=3
   thêm cặp (d→d): khớp
   thêm cặp (c→a): chưa ghép, +1
  đoạn [2,3] 'cd' đảo thành 'dc' so với 'da': f = 1
   thêm cặp (e→e): khớp
   thêm cặp (b→b): khớp
  đoạn [1,4] 'bcde' đảo thành 'edcb' so với 'edab': f = 1
   thêm cặp (f→f): khớp
   thêm cặp (a→c): ghép hoán đổi với (c→a)
  đoạn [0,5] 'abcdef' đảo thành 'fedcba' so với 'fedabc': f = 1


Ở ví dụ 1, DP chọn một cách chia khác với lời giải thích của đề (cả chuỗi là một đoạn, 4 thao tác: 1 hoán đổi `(b→c),(c→b)` + 3 lần thay) — cũng tối ưu; nhiều cách chia có cùng chi phí.

Với tâm `c = 5` (giữa chuỗi độ dài 6), khi phủ cả chuỗi thì `f = 1` ⇒ chi phí đảo `1 + 1 = 2` — đúng ví dụ 3.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minOperations(self, word1: str, word2: str) -> int:
        n = len(word1)
        A = [ord(ch) - 97 for ch in word1]
        B = [ord(ch) - 97 for ch in word2]
        cost = [[10**9] * (n + 1) for _ in range(n + 1)]

        for i in range(n):                       # đoạn không đảo
            un = [0] * 676
            f = 0
            for j in range(i, n):
                x, y = A[j], B[j]
                if x != y:
                    if un[y * 26 + x]:
                        un[y * 26 + x] -= 1
                    else:
                        un[x * 26 + y] += 1
                        f += 1
                cost[i][j + 1] = f

        for c in range(2 * n - 1):               # đoạn đảo, mở rộng từ tâm
            i, j = c // 2, (c + 1) // 2
            un = [0] * 676
            f = 0
            while i >= 0 and j < n:
                pairs = ((A[i], B[i]),) if i == j else ((A[j], B[i]), (A[i], B[j]))
                for x, y in pairs:
                    if x != y:
                        if un[y * 26 + x]:
                            un[y * 26 + x] -= 1
                        else:
                            un[x * 26 + y] += 1
                            f += 1
                if f + 1 < cost[i][j + 1]:
                    cost[i][j + 1] = f + 1
                i -= 1
                j += 1

        dp = [0] + [10**9] * n
        for j in range(1, n + 1):
            best = dp[j]
            for i in range(j):
                v = dp[i] + cost[i][j]
                if v < best:
                    best = v
            dp[j] = best
        return dp[n]

for a, b, exp in examples:
    got = Solution().minOperations(a, b)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
4 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Các đoạn độc lập:** thao tác chỉ tác động trong một đoạn, nên tổng chi phí = tổng chi phí từng đoạn ⇒ DP chia đoạn đúng (mọi cách chia đều được xét).
2. **Chi phí đoạn không đảo = `M − P`** (chứng minh ở mục 6: cận trên bằng xây dựng, cận dưới bằng đếm số vị trí được sửa trên mỗi thao tác).
3. **Đảo:** mỗi vị trí chỉ được đảo một lần mà phép đảo luôn chạm mọi vị trí của đoạn ⇒ mỗi đoạn đảo tối đa một lần. Các phép thay/hoán đổi trước khi đảo tương ứng một-một với các phép trên vị trí đối xứng sau khi đảo, nên có thể coi như đảo trước ⇒ chi phí `1 + f(rev(a), b)`. BFS vét cạn ở mục 12 xác nhận điều này.
4. **Tính bảng đúng:** ghép tham lam "gặp cặp ngược đang chờ thì ghép ngay" cho số cặp ghép bằng `min(cnt[x→y], cnt[y→x])` cho mỗi loại, nên `f` sau mỗi bước bằng `M − P` của đoạn hiện tại. Mở rộng từ tâm thêm đúng các cặp `(a[i + j − k], b[k])` của đoạn đảo.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(word1) ≤ 100`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS + chia đoạn | mũ theo độ dài đoạn | mũ |
| 2. DP + `Counter` mỗi đoạn | `O(n^3)` | `O(n)` |
| 3. Đệ quy có nhớ | `O(n^3)` | `O(n)` |
| 4. Bảng chi phí + DP | `O(n^2)` (với hằng số 676 cho việc khởi tạo mảng đếm: `O(n · 676)`) | `O(n^2)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [15]:
impls = {
    'cubic': min_ops_cubic,
    'memo': min_ops_memo,
    'fast': min_ops_fast,
    'submit': lambda a, b: Solution().minOperations(a, b),
}
edge = [
    ("a", "a", 0),
    ("a", "b", 1),
    ("ab", "ba", 1),          # một hoán đổi (hoặc một lần đảo)
    ("abc", "cba", 1),        # đảo cả đoạn
    ("aaaa", "bbbb", 4),
    ("abab", "baba", 1),      # đảo cả chuỗi: "abab" -> "baba"
    ("zzzz", "zzzz", 0),
]
for a, b, exp in examples + edge:
    for name, f in impls.items():
        got = f(a, b)
        assert got == exp, (name, a, b, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với BFS vét cạn

In [16]:
random.seed(3579)
cnt = 0
for _ in range(300):
    n = random.randint(1, 5)
    a = ''.join(random.choice('abc') for _ in range(n))
    b = ''.join(random.choice('abc') for _ in range(n))
    exp = min_ops_brute(a, b)
    for name, f in impls.items():
        assert f(a, b) == exp, (name, a, b)
    cnt += 1
print('Khớp BFS vét cạn trên', cnt, 'cặp chuỗi ngẫu nhiên.')

Khớp BFS vét cạn trên 300 cặp chuỗi ngẫu nhiên.


### 12.3. Đối chiếu các cách nhanh với nhau + ca lớn `n = 100`

In [17]:
import time
for _ in range(200):
    n = random.randint(1, 30)
    a = ''.join(random.choice('abcd') for _ in range(n))
    b = ''.join(random.choice('abcd') for _ in range(n))
    assert min_ops_cubic(a, b) == min_ops_fast(a, b) == Solution().minOperations(a, b)

for alpha in ['ab', 'abcde', 'abcdefghijklmnopqrstuvwxyz']:
    a = ''.join(random.choice(alpha) for _ in range(100))
    b = ''.join(random.choice(alpha) for _ in range(100))
    t = time.time(); x = Solution().minOperations(a, b); t1 = time.time() - t
    t = time.time(); y = min_ops_cubic(a, b); t2 = time.time() - t
    assert x == y
    print(f"|Σ|={len(alpha):>2}: đáp án {x:>3} | O(n^2) {t1:.3f}s | O(n^3) {t2:.3f}s")

|Σ|= 2: đáp án  18 | O(n^2) 0.003s | O(n^3) 0.042s
|Σ|= 5: đáp án  46 | O(n^2) 0.003s | O(n^3) 0.069s
|Σ|=26: đáp án  85 | O(n^2) 0.003s | O(n^3) 0.097s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm hoán đổi theo `min` số lần **xuất hiện ký tự** thay vì theo cặp lệch ngược nhau

In [18]:
def f_wrong_letters(a, b):
    mism = [(x, y) for x, y in zip(a, b) if x != y]
    ca = Counter(x for x, _ in mism)
    cb = Counter(y for _, y in mism)
    swaps = sum(min(ca[c], cb[c]) for c in ca) // 2       # SAI
    return len(mism) - swaps

a, b = "abc", "bca"      # chu trình độ dài 3: không có cặp ngược nhau
print('sai :', f_wrong_letters(a, b))   # 2
print('đúng:', f_segment(a, b))         # 3
print('chi phí đoạn thật (BFS, có thể đảo):', seg_bfs(a, b), '| cả bài (được chia/đảo):', min_ops_brute(a, b))

sai : 2
đúng: 3
chi phí đoạn thật (BFS, có thể đảo): 2 | cả bài (được chia/đảo): 2


**Vì sao sai:** `a→b, b→c, c→a` là chu trình độ dài 3; một hoán đổi chỉ sửa được 1 vị trí (vì vị trí kia vẫn lệch và không được hoán đổi lần 2). Chỉ cặp `(x→y)` + `(y→x)` mới sửa đôi được.

### Lỗi 2 — Quên lựa chọn đảo đoạn

In [19]:
def min_ops_no_reverse(w1, w2):
    n = len(w1)
    return partition_dp(n, lambda i, j: f_segment(w1[i:j], w2[i:j]))

print('sai :', min_ops_no_reverse("abcdef", "fedabc"))   # 6
print('đúng:', Solution().minOperations("abcdef", "fedabc"))  # 2

sai : 5
đúng: 2


**Vì sao sai:** đảo một đoạn dài có thể sửa rất nhiều vị trí chỉ với 1 thao tác.

### Lỗi 3 — Coi như mỗi vị trí được hoán đổi **nhiều lần** (chi phí kiểu "sắp xếp hoán vị")

In [20]:
def cost_wrong_multi(a, b):
    # SAI: coi như hoán đổi được tùy ý -> chi phí như sắp xếp hoán vị: #lệch - #chu trình
    # (chỉ xét khi a là hoán vị của b, các chữ cái khác nhau)
    if sorted(a) != sorted(b) or len(set(a)) != len(a):
        return f_segment(a, b)
    pos = {ch: i for i, ch in enumerate(b)}
    seen, cycles, mism = set(), 0, 0
    for i in range(len(a)):
        if a[i] != b[i]:
            mism += 1
            if i not in seen:
                cycles += 1
                j = i
                while j not in seen:
                    seen.add(j)
                    j = pos[a[j]]
    return mism - cycles

def min_ops_wrong3(w1, w2):
    return partition_dp(len(w1), lambda i, j: min(cost_wrong_multi(w1[i:j], w2[i:j]),
                                                  1 + cost_wrong_multi(w1[i:j][::-1], w2[i:j])))

random.seed(33)
found = None
for _ in range(3000):
    n = random.randint(3, 5)
    b = ''.join(random.sample('abcde', n))
    a = ''.join(random.sample(b, n))
    if min_ops_wrong3(a, b) != min_ops_brute(a, b):
        found = (a, b, min_ops_wrong3(a, b), min_ops_brute(a, b))
        break
print('Phản ví dụ (a, b, sai, đúng):', found)

Phản ví dụ (a, b, sai, đúng): ('bcade', 'aecdb', 3, 4)


**Vì sao sai:** đề giới hạn **mỗi vị trí** chỉ tham gia **một** lần hoán đổi, nên không thể sắp xếp chu trình dài bằng nhiều hoán đổi liên tiếp như bài "số swap tối thiểu để sắp xếp".

### Lỗi 4 — Khi đảo, so `a` với `reverse(b)` nhưng quên cộng 1

In [21]:
def cost_wrong_no_plus1(a, b):
    return min(f_segment(a, b), f_segment(a[::-1], b))          # SAI: thiếu +1

def min_ops_wrong4(w1, w2):
    return partition_dp(len(w1), lambda i, j: cost_wrong_no_plus1(w1[i:j], w2[i:j]))

print('sai :', min_ops_wrong4("abc", "cba"))      # 0
print('đúng:', Solution().minOperations("abc", "cba"))  # 1

sai : 0
đúng: 1


**Vì sao sai:** phép đảo cũng là một thao tác.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia mảng/chuỗi thành các đoạn, mỗi đoạn có chi phí riêng" ⇒ **DP chia đoạn** `O(n^2)` lần gọi chi phí.
- "Mỗi vị trí dùng tối đa một lần cho mỗi loại thao tác" ⇒ thường dẫn tới **ghép cặp** (matching) đơn giản, không phải sắp xếp hoán vị.
- Hoán đổi sửa **hai** vị trí chỉ khi hai vị trí là cặp `(x→y)`, `(y→x)`.
- Chi phí cho **mọi đoạn**: cố định đầu trái rồi mở rộng sang phải; với đoạn **đảo ngược**: cố định **tâm** rồi mở rộng hai phía (giống đếm palindrome).
- Luôn kiểm chứng công thức "tự suy ra" bằng một BFS vét cạn trên dữ liệu nhỏ.

## 15. Biến thể của bài

1. Không có phép đảo ⇒ chỉ còn `f(a, b)` cho cả chuỗi (chia đoạn không giúp gì vì `f` cộng tính... gần như).
2. Chi phí thao tác khác nhau (`cRep`, `cSwap`, `cRev`) ⇒ chi phí đoạn `min(...)` với trọng số; ghép cặp có lợi chỉ khi `cSwap < 2·cRep`.
3. Cho phép hoán đổi **không giới hạn** mỗi vị trí ⇒ bài sắp xếp hoán vị: `#lệch − #chu trình` (tối đa hóa số chu trình — khó hơn khi có ký tự lặp).
4. Bài cùng họ: LC 72 *Edit Distance*, LC 854 *K-Similar Strings*, LC 132 *Palindrome Partitioning II* (DP chia đoạn + mở rộng từ tâm).

In [22]:
# Biến thể 2: chi phí khác nhau
def min_ops_weighted(w1, w2, c_rep, c_swp, c_rev):
    def g(a, b):
        cnt = Counter((x, y) for x, y in zip(a, b) if x != y)
        M = sum(cnt.values())
        P = sum(min(v, cnt[(y, x)]) for (x, y), v in cnt.items() if x < y)
        if c_swp < 2 * c_rep:
            return P * c_swp + (M - 2 * P) * c_rep
        return M * c_rep
    return partition_dp(len(w1), lambda i, j: min(g(w1[i:j], w2[i:j]),
                                                  c_rev + g(w1[i:j][::-1], w2[i:j])))

print(min_ops_weighted("abcdf", "dacbe", 1, 1, 1), min_ops_weighted("abcdf", "dacbe", 1, 3, 5))

4 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | BFS thao tác + chia đoạn | mũ | mũ | Kiểm chứng công thức |
| 2 | DP + `Counter` mỗi đoạn | `O(n^3)` | `O(n)` | Đủ nhanh với `n ≤ 100`, dễ viết |
| 3 | Đệ quy có nhớ | `O(n^3)` | `O(n)` | Tư duy top-down |
| 4 | Bảng `fwd`/`rev` + DP | `O(n^2)` | `O(n^2)` | **Nộp bài**, mở rộng tốt |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `f_segment("abba", "baab")` = ?

**Bài 2 (tính tay):** `Solution().minOperations("abcd", "badc")` = ?

**Bài 3 (điền TODO):** viết `swap_pairs(a, b)` trả về `P`.

**Bài 4 (tự viết lại):** viết lại `cost_tables` chỉ dùng **một** bảng `best[i][j] = min(fwd, 1 + rev)`.

**Bài 5 (mini challenge):** khôi phục **cách chia đoạn** tối ưu và với mỗi đoạn cho biết có đảo hay không, rồi kiểm tra tổng chi phí.

In [23]:
def swap_pairs_todo(a, b):
    # TODO: đếm cnt[(x, y)] cho vị trí lệch, cộng min(cnt[x,y], cnt[y,x]) với x < y
    raise NotImplementedError

def best_table_todo(w1, w2):
    # TODO
    raise NotImplementedError

def reconstruct_todo(w1, w2):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Lệch ở cả 4 vị trí: `(a→b), (b→a), (b→a), (a→b)`. (2) Ghép được 2 cặp. (3) `4 − 2 = 2`.

**Bài 2.** (1) Chia `"ab" | "cd"`. (2) Mỗi đoạn là một hoán đổi (hoặc một lần đảo). (3) Đáp án 2.

**Bài 3.** (1) `Counter` trên vị trí lệch. (2) Chỉ cộng khi `x < y` để không đếm hai lần. (3) So với `M − f_segment`.

**Bài 4.** (1) Khởi tạo `best` bằng vô cực. (2) Điền từ vòng không đảo, cập nhật `min` trong vòng tâm. (3) Đó chính là lời giải nộp bài.

**Bài 5.** (1) Lưu `choice[j]`. (2) Lần ngược. (3) Cộng lại chi phí từng đoạn bằng `cost_segment`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
a1 = f_segment("abba", "baab")
a2 = Solution().minOperations("abcd", "badc")
assert a1 == 2 and a2 == 2
assert a2 == min_ops_brute("abcd", "badc")
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 2 | Bài 2: 2


In [25]:
# Bài 3
def swap_pairs(a, b):
    cnt = Counter((x, y) for x, y in zip(a, b) if x != y)
    return sum(min(v, cnt[(y, x)]) for (x, y), v in cnt.items() if x < y)

random.seed(3)
for _ in range(300):
    n = random.randint(1, 10)
    a = ''.join(random.choice('abc') for _ in range(n))
    b = ''.join(random.choice('abc') for _ in range(n))
    M = sum(x != y for x, y in zip(a, b))
    assert swap_pairs(a, b) == M - f_segment(a, b)
print('Bài 3: swap_pairs đúng')

Bài 3: swap_pairs đúng


In [26]:
# Bài 4
def best_table(w1, w2):
    fwd, rev = cost_tables(w1, w2)
    n = len(w1)
    return [[min(fwd[i][j], 1 + rev[i][j]) if j > i else 0 for j in range(n + 1)] for i in range(n + 1)]

random.seed(4)
for _ in range(100):
    n = random.randint(1, 8)
    a = ''.join(random.choice('abcd') for _ in range(n))
    b = ''.join(random.choice('abcd') for _ in range(n))
    bt = best_table(a, b)
    for i in range(n):
        for j in range(i + 1, n + 1):
            assert bt[i][j] == cost_segment(a[i:j], b[i:j])
print('Bài 4: best_table đúng')

Bài 4: best_table đúng


In [27]:
# Bài 5
def reconstruct(w1, w2):
    n = len(w1)
    dp = [0] + [10**9] * n
    ch = [None] * (n + 1)
    for j in range(1, n + 1):
        for i in range(j):
            c = cost_segment(w1[i:j], w2[i:j])
            if dp[i] + c < dp[j]:
                dp[j], ch[j] = dp[i] + c, i
    segs, j = [], n
    while j:
        i = ch[j]
        a, b = w1[i:j], w2[i:j]
        segs.append((a, b, 'đảo' if 1 + f_segment(a[::-1], b) < f_segment(a, b) else 'không đảo'))
        j = i
    return dp[n], segs[::-1]

for a, b, _ in examples:
    total, segs = reconstruct(a, b)
    assert total == sum(cost_segment(x, y) for x, y, _ in segs) == Solution().minOperations(a, b)
    print(total, segs)

4 [('abcdf', 'dacbe', 'không đảo')]
4 [('abceded', 'baecfef', 'không đảo')]
2 [('abcdef', 'fedabc', 'đảo')]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f(a, b) = #lệch - Σ_{x<y} min(cnt[x→y], cnt[y→x])     # thay + hoán đổi cặp ngược
cost(i, j) = min(f(a[i:j], b[i:j]), 1 + f(rev(a[i:j]), b[i:j]))
bảng fwd: cố định i, mở rộng j, ghép tham lam cặp ngược
bảng rev: cố định tâm c = i + j, mở rộng hai phía, thêm (a[j], b[i]) và (a[i], b[j])
dp[j] = min_i dp[i] + cost(i, j)
```

**Checklist:**

- [ ] Hoán đổi chỉ ghép cặp `(x→y)` với `(y→x)`?
- [ ] Đảo cộng thêm 1 và mỗi đoạn đảo tối đa một lần?
- [ ] Tâm lẻ (`i == j`) chỉ thêm một cặp?
- [ ] DP chia đoạn xét mọi `i < j`?
- [ ] Đã đối chiếu với BFS vét cạn trên chuỗi ngắn?